# HomeBot Gemma 4 Fine-tune (Unsloth, E2B / E4B, bf16 LoRA)

End-to-end notebook that takes the merged HomeBot multi-turn ChatML dataset,
fine-tunes **Gemma 4 E2B or E4B** with **bf16 LoRA** via Unsloth on a free
Colab T4, and exports a Q4_K_M GGUF that slots directly into the existing
Ollama-based DeepAgent runtime. It mirrors the `unsloth_qwen3_5_4b_homebot`
notebook so you can A/B the two architectures on the same dataset.

**Model size is a single knob** (`MODEL_SIZE` in step 0). Recommended flow:

1. First run: `MODEL_SIZE = "E2B"` (~12 min / 2 epochs on T4). Full bf16 LoRA,
   fits on T4 with ~4 GB free. Sanity-checks pipeline, chat template, prompts.
2. Final run: `MODEL_SIZE = "E4B"` on an **L4/A100/RTX 4090+** for bf16, OR on
   T4 with auto-QLoRA (4-bit base, bf16 adapters -- slight accuracy hit).

Everything else (LoRA attach, chat template, SFT loop, train-on-responses,
GGUF export, Modelfile emit) is identical across sizes.

**Why Gemma 4 vs Qwen 3.5?**

- Gemma 4 has native OpenAI-format tool calling built into the tokenizer,
  so the `tool_calls` / `tool_response` round-trip tokenises cleanly without
  the `<tool_call>` wrapper that Qwen ChatML needs.
- 128K native context (we only use ~12K), matters if you ever extend the
  training examples to include full multi-turn chat history.
- Multilingual (140 languages). For Indian-English + Hindi code-mixing on
  Telegram this is often a noticeable upgrade over Qwen.
- Same LoRA recipe -- drop-in replacement. Reuses the
  `kanakjr/homebot-qwen3.5` dataset unchanged because the ChatML conversations
  are model-agnostic until `apply_chat_template` runs in step 5.

**Training loop shape** (same as the Qwen notebook):

```
system -> user -> assistant+tool_calls -> tool -> ... -> assistant (final text)
```

We use `train_on_responses_only` with Gemma's `<start_of_turn>` delimiters so
loss only applies to `model` + tool-call spans; the model is NOT rewarded for
regurgitating the (long) HomeBot system prompt.

**Inputs required.**
1. A T4 (or better) Colab GPU runtime.
2. An HF **write** token pasted into step 0 below.
3. `MODEL_SIZE` set to `"E2B"` or `"E4B"` (step 0).

That's it -- `Runtime -> Run all` then go grab a coffee.


## 0. Configuration -- set your HF token + pick a model size

Paste your Hugging Face **write** token in the cell below, then flip
`MODEL_SIZE` to `"E2B"` or `"E4B"`. Everything else (dataset pull, GGUF
export repo name) is derived automatically.

| `MODEL_SIZE` | bf16 VRAM (load) | On T4 (16 GB) | ~2-epoch wall-clock |
| --- | --- | --- | --- |
| `"E2B"` | ~10 GB | bf16 LoRA fits comfortably | ~12 min |
| `"E4B"` | ~16 GB | **auto-QLoRA** (load_in_4bit) required | ~25 min (T4) / ~12 min (L4) |

Token + config lives in the cell below so you don't need Colab Secrets or an
`.env` file. Just remember to clear it before sharing the notebook.


In [ ]:
import os

# --- REQUIRED: paste your HF write token here -----------------------------
HF_TOKEN = ""  # e.g. "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
# --------------------------------------------------------------------------

# --- Model size toggle ----------------------------------------------------
# "E2B" -> ~10 GB bf16 VRAM, ~12 min / 2 epochs on T4. Recommended first run.
# "E4B" -> ~16 GB bf16 VRAM. On T4 the registry forces load_in_4bit=True
#          (QLoRA) so weights still fit; on L4/A100/4090+ flip it to False.
MODEL_SIZE = "E2B"   # "E2B" or "E4B"

_MODEL_REGISTRY = {
    "E2B": {
        "name":    "unsloth/gemma-4-E2B-it",
        "gguf":    "kanakjr/homebot-gemma4-e2b-gguf",
        "max_seq": 12288,
        # bf16 LoRA on T4: comfortable fit for E2B (~10 GB base + adapters).
        "load_in_4bit": False,
    },
    "E4B": {
        "name":    "unsloth/gemma-4-E4B-it",
        "gguf":    "kanakjr/homebot-gemma4-e4b-gguf",
        "max_seq": 12288,
        # E4B bf16 is 16 GB, which doesn't fit on T4's 14.5 GB. QLoRA keeps
        # the base in 4-bit NF4 and only trains bf16 adapters. Flip this to
        # False if you are on L4 / A100 / RTX 4090 / 6000 Ada / H100 / B200.
        "load_in_4bit": True,
    },
}
assert MODEL_SIZE in _MODEL_REGISTRY, f"MODEL_SIZE must be one of {list(_MODEL_REGISTRY)}"
_m = _MODEL_REGISTRY[MODEL_SIZE]
MODEL_NAME     = _m["name"]
MAX_SEQ_LENGTH = _m["max_seq"]
LOAD_IN_4BIT   = _m["load_in_4bit"]

# BUILD_TAG is threaded through every artifact path (LoRA dir, GGUF dir,
# Modelfile filename, Ollama model name) so E2B and E4B runs don't overwrite
# each other. e.g. "homebot-gemma4-e2b", "homebot-gemma4-e4b".
BUILD_TAG = f"homebot-gemma4-{MODEL_SIZE.lower()}"

# --- Dataset + (optional) model repo names --------------------------------
# Reuses the Qwen-trained dataset verbatim -- the conversations are plain
# role/content records, the model-specific chat template is only applied in
# step 5 via `apply_chat_template`.
HUB_DATASET_REPO = "kanakjr/homebot-qwen3.5"
HUB_MODEL_REPO   = _m["gguf"]

# --- Knobs you rarely touch -----------------------------------------------
REAL_OVERSAMPLE = 4       # step 5.5: how many times each real Telegram row is repeated
LORA_R = 16; LORA_ALPHA = 16  # bump both to 32 if under-fitting

# --- Propagate to env so downstream libs (datasets, huggingface_hub) see it
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# Best-effort Colab Secrets fallback (only used if you left HF_TOKEN blank).
if not HF_TOKEN:
    try:
        from google.colab import userdata
        _secret = userdata.get("HF_TOKEN") or ""
        if _secret:
            HF_TOKEN = _secret
            os.environ["HF_TOKEN"] = _secret
            os.environ["HUGGING_FACE_HUB_TOKEN"] = _secret
            print("[config] using HF_TOKEN from Colab Secrets")
    except Exception:
        pass

assert HF_TOKEN, (
    "HF_TOKEN is empty -- paste your token in the cell above, or add it as a "
    "Colab Secret named HF_TOKEN."
)
print(f"[config] HF_TOKEN set (len={len(HF_TOKEN)})")
print(f"[config] MODEL_SIZE    = {MODEL_SIZE}")
print(f"[config] base model    = {MODEL_NAME}")
print(f"[config] load_in_4bit  = {LOAD_IN_4BIT}")
print(f"[config] max seq len   = {MAX_SEQ_LENGTH}")
print(f"[config] build tag     = {BUILD_TAG}")
print(f"[config] dataset repo  = {HUB_DATASET_REPO}")
print(f"[config] gguf    repo  = {HUB_MODEL_REPO}")


## 1. Install Unsloth + dependencies

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv

# Colab ships torchcodec pre-built against its bundled torch; Gemma 4 pulls
# in timm which depends on torchcodec transitively, and the bundled .so
# fails to load against the unsloth-pinned torch with
# "undefined symbol: _ZN3c104cuda...". Uninstall first, then let the timm
# install pull a matching version below.
!pip uninstall -y -qqq torchcodec torchao || true

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

# Core deps locked to what Gemma 4 ships against in Unsloth's reference
# notebook (2026.4.x). Transformers 5.5.0 is required for Gemma 4 patching;
# timm is required for Gemma 4's vision/audio towers even though we only
# train the text path.
!uv pip install -qqq "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer sentencepiece protobuf
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install --no-deps transformers==5.5.0
!uv pip install --no-deps --upgrade timm
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

# Defensive: re-uninstall any torchcodec that crept back in as a transitive
# dep. Safe to skip if not present.
!pip uninstall -y -qqq torchcodec || true


## 2. Load the HomeBot dataset (fail-fast)

We pull the dataset **before** downloading Gemma 4 weights (~10 GB on E2B,
~16 GB on E4B) so any auth / repo / schema problem blows up in seconds
instead of after a full model download.

Default: HF Hub repo set in step 0 (`HUB_DATASET_REPO` -- reuses the
`kanakjr/homebot-qwen3.5` dataset unchanged because ChatML conversations are
model-agnostic). Flip `USE_HUB = False` in the cell below to upload
`qwen3_5_training.jsonl` + `qwen3_5_val.jsonl` from your laptop instead.


In [ ]:
from collections import Counter
from datasets import load_dataset, Dataset, DatasetDict

# Default path: pull the already-merged multi-turn dataset from the HF Hub
# repo set in step 0. Flip USE_HUB = False if you want to upload the two
# JSONLs from your laptop instead (e.g. iterating locally without pushing).
USE_HUB = True
LOCAL_TRAIN = "qwen3_5_training.jsonl"
LOCAL_VAL = "qwen3_5_val.jsonl"


def _load_hub_with_parquet_fallback(repo_id: str, token: str):
    """Try load_dataset() first. If Colab's `datasets` version is older than
    the one that pushed the parquet (seen as `TypeError` from
    `string_to_arrow` or `ValueError: Feature type 'Json' not found`), fall
    back to downloading raw parquet, materializing rows as Python dicts via
    `pq.to_pylist()`, and rebuilding with `Dataset.from_list()` so features
    are re-inferred locally. That bypasses the HF feature-metadata parser
    entirely and survives any future schema-type drift."""
    try:
        return load_dataset(repo_id, token=token)
    except (TypeError, ValueError) as e:
        print(f"[load] load_dataset() schema parse failed: {type(e).__name__}: {e}")
        print("[load] falling back to raw parquet via huggingface_hub...")
    from huggingface_hub import hf_hub_download, list_repo_files
    import pyarrow.parquet as pq

    files = list_repo_files(repo_id, repo_type="dataset", token=token)
    parquet_files = [f for f in files if f.endswith(".parquet")]
    if not parquet_files:
        raise RuntimeError(f"No parquet files found in {repo_id} (files: {files[:10]})")

    # push_to_hub() writes files like `data/train-00000-of-00001.parquet`.
    # Group by split name (the basename prefix before the first `-`).
    splits: dict[str, list[str]] = {}
    for path in parquet_files:
        basename = path.split("/")[-1]          # e.g. train-00000-of-00001.parquet
        split = basename.split("-")[0]          # -> "train"
        if split not in ("train", "validation", "test"):
            split = "train"                     # fallback for odd layouts
        splits.setdefault(split, []).append(path)

    import json as _json  # local import to avoid shadowing

    out: dict[str, Dataset] = {}
    for split, paths in splits.items():
        records = []
        for p in paths:
            lp = hf_hub_download(repo_id, filename=p, repo_type="dataset", token=token)
            # to_pylist() fully materializes rows as plain Python dicts,
            # dropping the HF feature-schema metadata stashed in the parquet
            # file. Dataset.from_list() then re-infers features using the
            # locally-installed datasets version, which is what we want.
            records.extend(pq.read_table(lp).to_pylist())

        # The newer `datasets` `Json` feature type serializes to parquet in
        # one of two shapes depending on the source data:
        #   (a) SCALAR:   whole value as a single JSON string per row
        #                 e.g. `messages` becomes "[{...}, {...}]"
        #   (b) PER-ELEM: list<string> where each element is itself
        #                 a JSON-encoded dict, e.g.
        #                 `messages` = ["{\"role\":\"user\",...}", ...]
        # We have to handle both when bypassing the HF feature parser.
        if records:
            json_scalar_cols = set()
            json_list_cols = set()
            for col in records[0].keys():
                for r in records:
                    v = r.get(col)
                    if isinstance(v, str) and v[:1] in ("[", "{"):
                        json_scalar_cols.add(col)
                        break
                    if (
                        isinstance(v, list)
                        and v
                        and isinstance(v[0], str)
                        and v[0][:1] in ("[", "{")
                    ):
                        json_list_cols.add(col)
                        break
            for col in sorted(json_scalar_cols):
                decoded = 0
                for r in records:
                    v = r.get(col)
                    if isinstance(v, str):
                        try:
                            r[col] = _json.loads(v)
                            decoded += 1
                        except _json.JSONDecodeError:
                            pass  # not JSON, leave as string
                if decoded:
                    print(
                        f"[load]   {split}.{col}: decoded {decoded} scalar JSON string(s) "
                        f"(shape: whole value per row)"
                    )
            for col in sorted(json_list_cols):
                decoded_rows = 0
                for r in records:
                    v = r.get(col)
                    if not isinstance(v, list):
                        continue
                    new_list = []
                    row_changed = False
                    for item in v:
                        if isinstance(item, str) and item[:1] in ("[", "{"):
                            try:
                                new_list.append(_json.loads(item))
                                row_changed = True
                                continue
                            except _json.JSONDecodeError:
                                pass
                        new_list.append(item)
                    if row_changed:
                        r[col] = new_list
                        decoded_rows += 1
                if decoded_rows:
                    print(
                        f"[load]   {split}.{col}: decoded per-element JSON strings in "
                        f"{decoded_rows} list(s) (shape: list<string> where each string "
                        f"is a JSON-encoded dict)"
                    )

        out[split] = Dataset.from_list(records)

        # Sanity: make sure `messages` survived round-trip as list[dict],
        # otherwise every downstream step (chat template, SFTTrainer) will
        # blow up in confusing ways.
        sample = out[split][0] if len(out[split]) > 0 else {}
        msgs = sample.get("messages")
        if msgs is not None and (
            not isinstance(msgs, list)
            or (msgs and not isinstance(msgs[0], dict))
        ):
            raise RuntimeError(
                f"[load] messages column in split '{split}' is not list[dict] after "
                f"fallback; type={type(msgs).__name__}, "
                f"inner={type(msgs[0]).__name__ if isinstance(msgs, list) and msgs else 'n/a'}. "
                f"Upgrade `datasets` in the install cell (e.g. `uv pip install -U datasets`) "
                f"so `load_dataset()` succeeds on the primary path."
            )

        print(f"[load] {split}: {len(out[split])} rows from {len(paths)} parquet file(s)")
    return DatasetDict(out)


if USE_HUB:
    ds = _load_hub_with_parquet_fallback(HUB_DATASET_REPO, HF_TOKEN)
else:
    if not (os.path.exists(LOCAL_TRAIN) and os.path.exists(LOCAL_VAL)):
        try:
            from google.colab import files  # noqa: F401
            print("Upload the two JSONL files when the file picker appears below...")
            uploaded = files.upload()  # expects qwen3_5_training.jsonl + qwen3_5_val.jsonl
            assert LOCAL_TRAIN in uploaded and LOCAL_VAL in uploaded, (
                f"Expected both {LOCAL_TRAIN} and {LOCAL_VAL}; got {list(uploaded)}"
            )
        except ImportError:
            raise RuntimeError(
                "Not running in Colab. Set USE_HUB=True, or place the two JSONLs in the CWD."
            )
    ds = load_dataset(
        "json",
        data_files={"train": LOCAL_TRAIN, "validation": LOCAL_VAL},
    )

print(ds)
train_sources = Counter(r.get("source", "?") for r in ds["train"])
val_sources = Counter(r.get("source", "?") for r in ds["validation"])
print(f"train sources: {dict(train_sources)}")
print(f"val   sources: {dict(val_sources)}")
print("\nTrain example keys:", list(ds["train"][0].keys()))
print("First user turn:", next(
    (m["content"][:140] for m in ds["train"][0]["messages"] if m["role"] == "user"),
    "(no user turn found)"
))


## 3. Load Gemma 4 ({MODEL_SIZE}) in bf16 LoRA mode

Gemma 4 is a unified VLM (text + image + audio). Our training data is text-only
so we load via `FastVisionModel` (the T4-safe path for VLMs), freeze the
vision + audio towers in step 4, and only attach LoRA on language layers.
`max_seq_length` **must** be passed here -- otherwise Unsloth silently caps
the sequence at 2048-4096 and most of our 12k-token rows get truncated past
the assistant turn, which `train_on_responses_only` then drops as all-labels
= -100.


In [ ]:
from unsloth import FastVisionModel
import torch

# Canonical loader from Unsloth's official Gemma 4 Colab notebook
# (https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma4_(E4B)-Vision.ipynb).
# Unsloth auto-detects T4 (no bf16) vs. Ampere+ (bf16) and picks the right
# 16-bit dtype for weights, LoRA adapters, and autocast. DO NOT pass `dtype=`
# manually -- the model registry drives the 4-bit / 16-bit choice via
# LOAD_IN_4BIT instead.
#
# IMPORTANT: we MUST pass `max_seq_length` here, not just on SFTConfig.
# Unsloth defaults this to 2048-4096 at load time and that cap silently
# clips downstream tokenisation even when SFTConfig.max_length is larger,
# causing `train_on_responses_only` to see truncated rows with no
# assistant response and drop them as all-labels=-100.
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = LOAD_IN_4BIT,
    use_gradient_checkpointing = "unsloth",
)


## 4. Attach LoRA adapters (text-only)

`r=16, lora_alpha=16` is a solid Gemma 4 starting point. If you see under-fitting after 2 epochs, bump both to 32.

We use `FastVisionModel.get_peft_model(...)` with `finetune_vision_layers=False` -- the HomeBot dataset is 100% text (Telegram + synthetic chat), so there is no signal for the vision or audio towers. Everything else (attention, MLP, language layers) gets LoRA'd.


In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # text-only dataset -> freeze vision tower
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## 5. Apply the gemma-4 chat template

The `gemma-4` template wraps turns as `<start_of_turn>role\ncontent<end_of_turn>`. Note Gemma uses `model` (not `assistant`) as the assistant role, and the current Unsloth template supports `system` and `tool` roles via the OpenAI-style tool-calling patch (HF PR #45257). Our training JSONL already has `role=system/user/assistant/tool` with OpenAI-style `tool_calls`, so `apply_chat_template` handles everything -- the `assistant` -> `model` rename happens inside the template.


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)


## 5.5 Oversample real Telegram rows

The merge step flagged a ~13:1 synthetic:real ratio. Real Telegram chats are
the most authentic signal for how the user actually talks (short, elliptical,
often no greeting), so we duplicate them `REAL_OVERSAMPLE` times inside the
train split. This is a "poor-man's sample weights" -- simpler to reason about
than trainer-level weighting, and it works cleanly with `train_on_responses_only`.

Rule of thumb: set `REAL_OVERSAMPLE` so the effective ratio ends up ~3:1 or
tighter. With 204 synthetic / 16 real in train, `REAL_OVERSAMPLE=4` gives
64 real, i.e. ~3.2:1.

In [ ]:
from datasets import concatenate_datasets

# REAL_OVERSAMPLE comes from step 0 (Configuration). Default 4.
real_rows = ds["train"].filter(lambda r: r.get("source") == "telegram")
synth_rows = ds["train"].filter(lambda r: r.get("source") != "telegram")

if len(real_rows) == 0:
    print("[oversample] no real rows found; skipping")
    train_balanced = ds["train"]
else:
    real_repeated = concatenate_datasets([real_rows] * REAL_OVERSAMPLE)
    train_balanced = concatenate_datasets([synth_rows, real_repeated]).shuffle(seed=3407)
    print(
        f"[oversample] real rows {len(real_rows)} -> {len(real_repeated)} "
        f"({REAL_OVERSAMPLE}x); synth unchanged at {len(synth_rows)}; "
        f"final train size = {len(train_balanced)}"
    )

ds_train = train_balanced
ds_val = ds["validation"]
print(f"train size={len(ds_train)}, val size={len(ds_val)}")

## 6. Render each conversation through the chat template

Gemma 4's `apply_chat_template` enforces **strict user/assistant alternation**
and raises `TemplateError: Conversation roles must alternate user/assistant/...`
the moment it sees a `system` or `tool` role. (Transformers PR #45257 adds
OpenAI-shape tool calling to the Gemma template but is not in the
`transformers==5.5.0` pin we use for Gemma 4 patching.)

Our HomeBot dataset has:

```
system -> user -> assistant+tool_calls -> tool -> assistant (final)
```

So before calling `apply_chat_template` we **flatten** the conversation into
strict `user/assistant/user/assistant` alternation:

- `system` content is prepended to the next `user` turn as plain text.
- `assistant.tool_calls` are serialised inline as
  `<tool_call>{...}</tool_call>` inside the assistant `content`. The model
  learns to emit these verbatim; Ollama's Gemma renderer re-wraps them as
  native tool calls at inference time.
- `role=tool` outputs become `user` turns wrapped in
  `<tool_response>...</tool_response>`. This preserves the alternation
  (previous assistant -> new user) and matches what Ollama's Gemma template
  produces when a tool result is injected at inference time.
- Accidental consecutive same-role turns are merged with `\n\n`.

This keeps the GGUF export compatible with Ollama's built-in Gemma template
while letting us train with the `transformers==5.5.0` Jinja template that
ships with Gemma 4's checkpoint.


In [ ]:
import json


def _flatten_to_strict_alternation(messages):
    """Rewrite HomeBot ChatML into strict user/assistant alternation so
    Gemma 4's chat template accepts it. Gemma-specific; the Qwen notebook
    keeps the native system/tool roles because `qwen3-instruct` understands
    them.
    """
    out = []
    pending_system = None

    def _serialize_tool_call(tc):
        fn = tc.get("function", {})
        name = fn.get("name", "")
        args = fn.get("arguments", "")
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except Exception:
                pass
        return json.dumps({"name": name, "arguments": args}, ensure_ascii=False)

    for m in messages:
        role = m.get("role")
        content = m.get("content") or ""

        if role == "system":
            pending_system = content
            continue

        if role == "user":
            if pending_system:
                content = f"{pending_system}\n\n{content}"
                pending_system = None
            out.append({"role": "user", "content": content})
            continue

        if role == "assistant":
            parts = []
            if content:
                parts.append(content)
            for tc in m.get("tool_calls") or []:
                parts.append(f"<tool_call>\n{_serialize_tool_call(tc)}\n</tool_call>")
            out.append({"role": "assistant", "content": "\n".join(parts)})
            continue

        if role == "tool":
            out.append({
                "role": "user",
                "content": f"<tool_response>\n{content}\n</tool_response>",
            })
            continue
        # Unknown role -- skip defensively rather than crash the whole run.

    # Merge accidental consecutive same-role turns (e.g. two `user` in a row
    # when a nudge follows a tool_response) so the template stays happy.
    collapsed = []
    for m in out:
        if collapsed and collapsed[-1]["role"] == m["role"]:
            collapsed[-1]["content"] = collapsed[-1]["content"] + "\n\n" + m["content"]
        else:
            collapsed.append(m)

    # Gemma also rejects a leading assistant turn; drop it if it ever slips
    # through (shouldn't happen with our dataset but defensive).
    while collapsed and collapsed[0]["role"] == "assistant":
        collapsed.pop(0)

    return collapsed


def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            _flatten_to_strict_alternation(convo),
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]
    return {"text": texts}

# CRITICAL: drop every non-text column with `remove_columns=...`. If we leave
# `messages` in, SFTTrainer's default collator tries to tensorize the nested
# list-of-dicts and crashes with "Could not infer dtype of dict".
keep_only_text = lambda ds: ds.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=ds.column_names,
)

train_ds = keep_only_text(ds_train)
val_ds = ds_val
if val_ds is not None and len(val_ds) > 0:
    val_ds = keep_only_text(val_ds)

print(f"train_ds size={len(train_ds)}  val_ds size={0 if val_ds is None else len(val_ds)}")
print(f"train_ds columns after render: {train_ds.column_names}  (should be ['text'])")
print("\n--- Rendered example (truncated) ---\n")
print(train_ds[0]["text"][:2000])


## 7. Configure SFTTrainer

Small batch + high grad-accum fits Gemma 4 E2B bf16 LoRA on a 16 GB T4 with
headroom to spare (~10 GB base + ~0.5 GB adapters + ~2 GB activations for a
12k-token sequence). For E4B with `LOAD_IN_4BIT=True` the base drops to ~6 GB
so you can raise `per_device_train_batch_size` to 2 if you want. `max_length
= MAX_SEQ_LENGTH` covers the HomeBot system prompt + full multi-turn
conversation with some headroom.


In [ ]:
from trl import SFTTrainer, SFTConfig

# FastVisionModel must be toggled into "training" mode before wrapping in
# SFTTrainer (see Unsloth's official 4B notebook). This is also what flips
# the vision encoder frozen + language-layer autocast bookkeeping.
FastVisionModel.for_training(model)

# Use the inner text tokenizer rather than the VLM processor for SFT.
# Defensive: the processor's __call__ treats the first positional arg as
# images (same quirk that crashes step 12 sanity inference with
# `ValueError: Incorrect image source`). Our training data is text-only
# so extracting the text tokenizer sidesteps that whole class of bugs.
# No-op on FastLanguageModel (plain tokenizer has no `.tokenizer`).
_text_tok = getattr(tokenizer, "tokenizer", tokenizer)

trainer = SFTTrainer(
    model = model,
    tokenizer = _text_tok,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        max_length = MAX_SEQ_LENGTH,
        packing = False,
        eval_strategy = "epoch" if val_ds is not None else "no",
    ),
)


## 8. `train_on_responses_only` -- loss ONLY on `model` + tool_calls

Without this, the model learns to regurgitate the long system prompt, which is wasted capacity. Gemma 4 ChatML delimiters are:

- `<start_of_turn>user\n` ... `<end_of_turn>`
- `<start_of_turn>model\n` ... `<end_of_turn>`


In [ ]:
from unsloth.chat_templates import train_on_responses_only

# `train_on_responses_only` tokenises `instruction_part` and
# `response_part` to locate loss-masking boundaries. Pass the inner text
# tokenizer so it matches whatever SFTTrainer used to tokenise the rows
# (we also set `tokenizer=_text_tok` on the trainer above). Also keeps us
# clear of the processor's image-as-positional-arg quirk.
# No-op on FastLanguageModel (plain tokenizer has no `.tokenizer`).
_text_tok = getattr(tokenizer, "tokenizer", tokenizer)

trainer = train_on_responses_only(
    trainer,
    tokenizer        = _text_tok,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)


## 9. Memory snapshot before training

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved (before training).")


## 10. Train

In [ ]:
trainer_stats = trainer.train()


## 11. Memory snapshot after training

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_training = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
training_percentage = round(used_memory_training / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']:.1f} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_training} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage}%.")
print(f"Peak reserved memory for training % of max memory = {training_percentage}%.")


## 12. Sanity inference on held-out HomeBot prompts

These should emit a tool call (structured `tool_calls` array) when the right tool is obvious. Run 2-3 variations per skill family to eyeball that the fine-tune preserved tool-calling behavior.

In [ ]:
from transformers import TextStreamer

FastVisionModel.for_inference(model)

# Each row tests a different skill family the dataset actually covered.
# "Healthy" output means a `<tool_call>{...}</tool_call>` block with the
# expected entity_id, followed by a short natural-language summary.
SANITY_PROMPTS = [
    # Home Assistant device control
    "turn off the air purifier",
    "dim the bedroom light to 40%",
    # Sensor query + synthesis rule
    "what's the temperature in the bedroom?",
    # Media pipeline
    "add the movie Dune Part Two",
    "search jellyseerr for succession",
    # Persistent memory
    "remember that my standing desk is switch.monitor_plug",
    "what did I say about deep work hours?",
    # Network admin
    "which devices are online right now?",
    # Obsidian / notes
    "summarize my notes from yesterday",
    # Interactive UI
    "what are my 3 favourite lights?",
]

# Matches the Modelfile fallback system prompt so behaviour at sanity-check
# time is close to what DeepAgent sends at runtime. At serve time the
# DeepAgent injects the canonical get_system_prompt() which supersedes this.
SYSTEM_PROMPT = (
    "You are HomeBotAI, an intelligent smart-home assistant powered by Home "
    "Assistant. The home is in India (IST timezone). Resident: Kanak.\n\n"
    "You have access to tools for Home Assistant control, media management "
    "(Sonarr/Radarr/Jellyfin/Jellyseerr), network admin (Deco mesh), "
    "Obsidian vault + persistent memory, link processing, interactive "
    "choices, and shell execution.\n\n"
    "Rules: be efficient with tool calls (1-3), summarize results in one "
    "short line, use colloquial names (never raw entity_ids), and stop "
    "after confirming an action."
)

for prompt in SANITY_PROMPTS:
    print("\n" + "=" * 60)
    print(f"USER: {prompt}")
    print("=" * 60)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    # Route through the same flattener we use at training time -- Gemma's
    # chat template rejects the `system` role, so we fold it into the user
    # turn to match what the model actually saw in step 6.
    text = tokenizer.apply_chat_template(
        _flatten_to_strict_alternation(messages),
        tokenize = False,
        add_generation_prompt = True,
    )
    # NB: loading Gemma 4 via FastVisionModel (the T4 dtype workaround)
    # makes `tokenizer` a Gemma-4 VLM *processor*. Calling a processor
    # positionally binds the first arg to `images`, which triggers
    # `load_image()` -> tries to base64-decode our chat template string
    # and raises `Incorrect image source` / `Incorrect padding`.
    # Fall through to the inner text tokenizer so plain text tokenises
    # cleanly; this is a no-op on FastLanguageModel (no .tokenizer attr).
    _text_tok = getattr(tokenizer, "tokenizer", tokenizer)
    inputs = _text_tok(text, return_tensors="pt").to("cuda")
    _ = model.generate(
        **inputs,
        max_new_tokens = 512,
        # Gemma 4 sampling defaults (matches Google's model card).
        temperature = 0.6, top_p = 0.9, top_k = 40,
        streamer = TextStreamer(tokenizer, skip_prompt=True),
    )


## 13. Save LoRA adapters locally (checkpoint)

In [ ]:
LORA_DIR = f"{BUILD_TAG}-lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"Saved LoRA adapters to {LORA_DIR}/")


## 14. Export merged model to GGUF Q4_K_M for Ollama

This clones `llama.cpp`, merges the LoRA into a 16-bit model, and quantizes to Q4_K_M. Produces `{BUILD_TAG}.Q4_K_M.gguf` (~1.5 GB for 2B, ~2.5 GB for 4B). Download via the Colab file browser, then on the Mac run the command printed by the next cell:

```bash
ollama create <BUILD_TAG> -f <BUILD_TAG>.Modelfile
```

In [ ]:
model.save_pretrained_gguf(
    BUILD_TAG,
    tokenizer,
    quantization_method = "q4_k_m",
)
GGUF_FILENAME = f"{BUILD_TAG}.Q4_K_M.gguf"
print(f"GGUF written to {BUILD_TAG}/ -- look for {GGUF_FILENAME}")


## 15. (Optional) Push GGUF + merged 16-bit weights to HF Hub

Flip the flags below if you want a reproducible uploaded artifact. Uses
`HF_TOKEN` and `HUB_MODEL_REPO` from step 0. Off by default -- the primary
delivery channel is a local Ollama import of the GGUF file from step 14.

In [ ]:
PUSH_GGUF_TO_HUB   = True
PUSH_MERGED_TO_HUB = False

# HUB_MODEL_REPO and HF_TOKEN both come from step 0.
if PUSH_GGUF_TO_HUB:
    model.push_to_hub_gguf(
        HUB_MODEL_REPO,
        tokenizer,
        quantization_method = ["q4_k_m"],
        token = HF_TOKEN,
    )
    print(f"Pushed GGUF to https://huggingface.co/{HUB_MODEL_REPO}")

if PUSH_MERGED_TO_HUB:
    model.push_to_hub_merged(
        HUB_MODEL_REPO + "-merged16",
        tokenizer,
        save_method = "merged_16bit",
        token = HF_TOKEN,
    )
    print(f"Pushed merged 16-bit weights to https://huggingface.co/{HUB_MODEL_REPO}-merged16")


## 16. Emit a ready-to-paste Ollama Modelfile

Writes `{BUILD_TAG}.Modelfile` next to the GGUF (e.g. `homebot-gemma4-e2b.Modelfile`). Download both, then on your Mac run `ollama create {BUILD_TAG} -f {BUILD_TAG}.Modelfile`. The DeepAgent's `_resolve_model` accepts `ollama:<name>`, so point it at `ollama:homebot-gemma4-e2b` or `ollama:homebot-gemma4-e4b` depending on which build you shipped.


In [ ]:
# Gemma 4 Ollama Modelfile. We rely on Ollama's built-in chat-template
# auto-detection from the GGUF metadata (the tokenizer's chat template is
# serialised inside the GGUF), so we don't ship a TEMPLATE block -- Gemma
# 4's template is complex (tool-calling PR #45257) and manually re-rendering
# it is a footgun. If your Ollama version ever ships without auto-detect,
# paste the template from `ollama show --modelfile gemma3` as a starting
# point and swap `assistant` -> `model`.
# The SYSTEM block is only a FALLBACK -- at runtime the DeepAgent injects
# the canonical get_system_prompt() which supersedes this.
MODELFILE_TEMPLATE = f'''FROM ./{BUILD_TAG}.Q4_K_M.gguf''' + r'''

# Gemma 4 sampling defaults (matches Google's model card).
PARAMETER temperature 0.6
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER repeat_penalty 1.05
PARAMETER num_ctx 8192
PARAMETER stop "<end_of_turn>"
PARAMETER stop "<start_of_turn>"
PARAMETER stop "<eos>"

SYSTEM """You are HomeBotAI, an intelligent smart-home assistant powered by Home Assistant.
The home is in India (IST timezone). Resident: Kanak.

You have access to tools for:
- Home Assistant device control (ha_call_service, ha_get_states, ha_search_entities)
- Media management (sonarr_*, radarr_*, jellyfin_*, jellyseerr_*, prowlarr_*, transmission_*)
- Network admin (deco_list_clients, deco_list_mesh_nodes, deco_reboot_nodes, deco_reservation_help)
- Obsidian vault + persistent memory (obsidian_*, memory_*)
- Link processing (process_and_save_link)
- Interactive choices (offer_choices -- tap-able buttons; end your turn after calling it)
- Shell execution (execute)

Rules:
1. Be efficient with tool calls -- 1-3 targeted calls over exhaustive searching.
2. Always provide a short natural-language summary after tool calls.
3. Use colloquial names in replies (e.g. "the purifier"), never raw entity_ids.
4. For short ordinal replies like "3" or "the second one", resolve against your
   previous message.
5. Confirm actions in one line and stop -- no filler tails, no second-guessing.
6. Synthesize redundant sensor readings (within ~1C / 5%RH) into one value instead
   of dumping raw lists.
"""
'''

MODELFILE_PATH = f"{BUILD_TAG}.Modelfile"
with open(MODELFILE_PATH, "w") as f:
    f.write(MODELFILE_TEMPLATE)
print(f"Wrote {MODELFILE_PATH} ({len(MODELFILE_TEMPLATE)} bytes)")

_gguf_name = f"{BUILD_TAG}.Q4_K_M.gguf"
_gguf_size = "~2.8 GB" if MODEL_SIZE == "E2B" else "~4.6 GB"
print("\n=== Next steps (run on your Mac, NOT in Colab) ===")
print("1. Download both artifacts from the Colab file browser:")
print(f"      {_gguf_name}  ({_gguf_size})")
print(f"      {MODELFILE_PATH}")
print("2. Put them in the same directory on the Mac.")
print(f"3. ollama create {BUILD_TAG} -f {MODELFILE_PATH}")
print("4. Quick sanity test:")
print(f"      ollama run {BUILD_TAG} 'turn off the air purifier'")
print("5. Point DeepAgent at the new model:")
print(f"      MODEL=ollama:{BUILD_TAG}")
print("   Then restart the DeepAgent server.")


---

**Troubleshooting**

- *`from unsloth import FastVisionModel` fails with `OSError: libavutil.so.*` or `undefined symbol: _ZN3c104cuda29c10_cuda_check_implementationEiPKcS2_jb`*: Colab's pre-installed `torchcodec` is built against a different PyTorch build than the one the install cell pins. Fix in-place with `!pip uninstall -y torchcodec`, then re-run the import cell. **Runtime restart NOT required.**
- *`RuntimeError: expected mat1 and mat2 to have the same dtype, but got: c10::BFloat16 != c10::Half`*: you loaded Gemma 4 with `FastLanguageModel` instead of `FastVisionModel`. Gemma 4 is a unified VLM; only the vision loader wires up dtypes correctly on T4 (you will see `Bfloat16 = FALSE -> Switching to 16bit LoRA` in the banner). Re-run step 3 + step 4 after confirming they use `FastVisionModel`.
- *OOM on T4 with `MODEL_SIZE = "E4B"`*: this is expected in bf16 (16 GB weights > 14.5 GB T4 VRAM). The registry in step 0 already forces `LOAD_IN_4BIT = True` for E4B; if you edited that, either flip it back or downgrade to `MODEL_SIZE = "E2B"`.
- *`transformers` version mismatch / `KeyError: 'gemma4'` inside `modeling_auto`*: the install cell pins `transformers==5.5.0`; restart the runtime if you installed an earlier version in this session.
- *`ImportError: cannot import name 'gemma4' from 'timm'`*: step 1's `pip install --no-deps --upgrade timm` didn't stick. Re-run it explicitly: `!pip install --no-deps --upgrade timm`, then re-run step 3.
- *`TemplateError: Conversation roles must alternate user/assistant/user/assistant/...`*: Gemma 4's stock Jinja chat template rejects `system` and `tool` roles (the OpenAI-tool-calling patch HF #45257 is not in `transformers==5.5.0`). Step 6 already works around this by flattening our multi-turn ChatML into strict user/assistant alternation via `_flatten_to_strict_alternation`. If you see this error, you are probably calling `tokenizer.apply_chat_template(convo, ...)` directly on the raw `messages`; route it through `_flatten_to_strict_alternation(convo)` first.
- *`Unsloth: Removed N out of M samples from train_dataset where all labels were -100`*: see the **Troubleshooting** section of `finetuning/README.md` (same root causes as the Qwen 3.5 notebook -- the primary one is forgetting `max_seq_length` on `FastVisionModel.from_pretrained`).
- *Low eval loss but bad tool calls*: inspect the rendered example in step 6 -- make sure tool messages show up as `<start_of_turn>tool ...`. If they collapse into the user turn, re-run step 5 and confirm the chat template resolved to `"gemma-4"` rather than silently falling back to the base tokenizer template.
- *Model only replies in natural language and never calls a tool*: you probably skipped step 8 (`train_on_responses_only`). Without it the model learns to reproduce the system prompt, crowding out tool-call patterns.
- *Tool calls fire but summary is verbose/robotic*: model is over-fitting the simulator style. Increase `REAL_OVERSAMPLE` in step 5.5 (try 6 or 8), or re-run `./run_pipeline.sh real --days 365 --limit 5000` locally to pull more genuine chats and re-merge.
- *GGUF export fails with "unsupported architecture"*: Unsloth's `save_pretrained_gguf` needs the merged 16-bit model. Re-run step 14; if it still fails, bypass via `model.save_pretrained_merged(...)` and quantize manually with `llama.cpp/convert-hf-to-gguf.py`.

**Iteration recipe**

1. Download the GGUF + Modelfile, `ollama create`, try 5-10 real Telegram-style messages.
2. If tool arguments are wrong (wrong entity_id, missing field), that usually means the dataset is too small for that skill. Run `./run_pipeline.sh generate` to add more synthetic queries for that skill family, then `simulate` a handful and re-merge.
3. If the reply text style is off but tool calls are right, oversample real data more aggressively (step 5.5) rather than adding more synthetic rows.
